# Consolidated Microsoft Agent Framework Showcase

A single notebook that exercises every MAF feature demonstrated across the three companion notebooks, wired into one `FoundryChatClient.as_agent()` instance with a single shared session.

## Features in this notebook

| Capability | How |
|---|---|
| Local Python `@tool` functions | inline `get_weather` / `get_current_time` / `get_current_location` + 6 imported ticket-management tools |
| Public MCP server | `MCPStreamableHTTPTool` → Microsoft Learn (`https://learn.microsoft.com/api/mcp`) |
| Auth-protected MCP server | `MCPStreamableHTTPTool` → MAGIC v22 with `Authorization: Bearer …` via custom `httpx.AsyncClient` |
| FoundryAgent-as-a-tool | hosted `travel-support-agent` v2 wrapped via `agent.as_tool(...)` |
| Persistent chat history | inline `SqliteHistoryProvider(HistoryProvider)` (in-memory cache + SQLite flush) |
| Middleware stack | `InputGuardrailMiddleware` → `ExceptionHandlingAgentMiddleware` → `LoggingAgentMiddleware` |
| Streaming + non-streaming | scenarios mix `async for chunk in agent.run(..., stream=True)` and `await agent.run(...)` |

## Prerequisites
1. `az login` completed (used by `AzureCliCredential`).
2. `.env` at the workspace root with: `AZURE_AI_PROJECT_ENDPOINT`, `AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME`, `API_KEY`.
3. The MAGIC v22 MCP server running at `http://localhost:9898/mcp` (Scenario D will skip itself if not reachable).
4. The hosted Foundry agent named `travel-support-agent` version `2` available on the project.

## 1. Setup — imports, env, paths, DB init

In [1]:
import os
import re
import sys
import json
import time
import logging
import socket
from collections.abc import Awaitable, Callable, Sequence
from datetime import datetime
from pathlib import Path
from random import randint
from typing import Annotated, Any

import httpx
from dotenv import load_dotenv
from pydantic import Field
from sqlalchemy.exc import SQLAlchemyError

from agent_framework import (
    AgentContext,
    AgentMiddleware,
    AgentResponse,
    HistoryProvider,
    MCPStreamableHTTPTool,
    Message,
    MiddlewareTermination,
    tool,
)
from agent_framework.foundry import FoundryAgent, FoundryChatClient
from azure.identity.aio import AzureCliCredential

load_dotenv(override=True)

# Make the lib/ package importable
LIB_PATH = str(Path("./lib").resolve())
if LIB_PATH not in sys.path:
    sys.path.insert(0, LIB_PATH)

from ticket_management.database import init_db
from ticket_management.chat_history_models import (
    BlockedQuery,
    ChatMessage,
    get_chat_session,
    init_chat_history_db,
)
from ticket_management.tools import (
    close_ticket,
    get_tickets_by_registered_by,
    get_tickets_by_resolved_by,
    register_ticket,
    resolve_ticket,
    search_tickets,
)

init_db()
init_chat_history_db()

LOGS_DIR = Path("../logs").resolve()
LOGS_DIR.mkdir(parents=True, exist_ok=True)
LOG_FILE = LOGS_DIR / "agent.log"
BLACKLIST_PATH = Path("../db/blacklist.txt").resolve()

print("Imports loaded.")
print(f"Log file       : {LOG_FILE}")
print(f"Blacklist file : {BLACKLIST_PATH}")

<frozen abc>:106: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.
<frozen abc>:106: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.


Imports loaded.
Log file       : C:\000 - MS - AGENTIC - AI\e2e\logs\agent.log
Blacklist file : C:\000 - MS - AGENTIC - AI\e2e\db\blacklist.txt


## 2. Local `@tool` Python functions

Three inline utility tools (weather, current time, current location) plus six ticket-management tools imported from `lib/ticket_management/tools.py`.

In [2]:
@tool(approval_mode="never_require")
def get_weather(
    location: Annotated[str, Field(description="The location to get the weather for.")],
) -> str:
    """Get the weather for a given location."""
    conditions = ["sunny", "cloudy", "rainy", "stormy"]
    return (
        f"The weather in {location} is {conditions[randint(0, 3)]} "
        f"with a high of {randint(20, 38)}\u00b0C."
    )


@tool(approval_mode="never_require")
def get_current_time(
    location: Annotated[
        str,
        Field(description="Free-form location label, e.g. 'Bengaluru' or 'New York'."),
    ],
) -> str:
    """Return the current local server time alongside the requested location label."""
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    return f"The current time (server clock) is {now} for location '{location}'."


@tool(approval_mode="never_require")
def get_current_location() -> str:
    """Return the agent's best-effort assumed user location (demo stub)."""
    return "User is currently in Bengaluru, India."


print("Utility tools defined: get_weather, get_current_time, get_current_location")

Utility tools defined: get_weather, get_current_time, get_current_location


In [3]:
ticket_tools = [
    register_ticket,
    get_tickets_by_registered_by,
    get_tickets_by_resolved_by,
    search_tickets,
    resolve_ticket,
    close_ticket,
]

print(f"Ticket tools loaded: {len(ticket_tools)}")

Ticket tools loaded: 6


## 3. MCP server tools

- **MS Learn** — public, no auth.
- **MAGIC v22** — protected; we attach an `Authorization: Bearer …` header via a custom `httpx.AsyncClient`.

In [4]:
ms_learn_tool = MCPStreamableHTTPTool(
    name="Microsoft Learn MCP Tool",
    url="https://learn.microsoft.com/api/mcp",
)
print("\u2713 MS Learn MCP tool ready")

✓ MS Learn MCP tool ready


In [5]:
MAGIC_V22_URL = "http://localhost:9898/mcp"
api_key = os.getenv("API_KEY", "")

magic_v22_tool = MCPStreamableHTTPTool(
    name="MAGIC v22 Orders and Complaints Tool",
    url=MAGIC_V22_URL,
    http_client=httpx.AsyncClient(
        headers={"Authorization": f"Bearer {api_key}"},
        timeout=30.0,
    ),
)


def _magic_v22_reachable(host: str = "localhost", port: int = 9898, timeout: float = 1.0) -> bool:
    """Quick TCP probe so Scenario D can self-skip if the server is down."""
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False


MAGIC_V22_AVAILABLE = _magic_v22_reachable()
print("\u2713 MAGIC v22 MCP tool ready")
print(f"  reachability probe: {'reachable' if MAGIC_V22_AVAILABLE else 'NOT reachable — Scenario D will be skipped'}")

✓ MAGIC v22 MCP tool ready
  reachability probe: reachable


## 4. Hosted Foundry agent as a tool

Reference an existing Foundry-hosted agent (`travel-support-agent` v2) and expose it via `as_tool(...)` so the consolidated agent can delegate travel questions to it.

In [6]:
travel_child_agent = FoundryAgent(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    agent_name="travel-support-agent",
    agent_version="2",
    credential=AzureCliCredential(),
)

travel_agent_tool = travel_child_agent.as_tool(
    name="travel-support-agent-tool",
    description=(
        "Authoritative travel knowledge base. MUST be called for ANY question about "
        "hotels, flights, travel packages, destinations, itineraries, tour operators, "
        "or travel agencies (e.g. Margie's Travels). Returns grounded data."
    ),
    arg_name="query",
    arg_description="The user's full travel-related question, passed verbatim.",
)

print("\u2713 travel-support-agent (v2) wrapped as a tool")

✓ travel-support-agent (v2) wrapped as a tool


## 5. Persistent chat history — `SqliteHistoryProvider`

Hybrid strategy:
- `get_messages` returns `state["messages"]` if cached, else cold-loads from the `chat_messages` table and hydrates the cache.
- `save_messages` appends to the in-memory cache **and** flushes new rows to SQLite.

Storage is the dedicated `db/chat_history.db` exposed by `ticket_management.chat_history_models`.

In [7]:
class SqliteHistoryProvider(HistoryProvider):
    """History provider that caches in state['messages'] and flushes to SQLite on save."""

    def __init__(self) -> None:
        super().__init__("sqlite-history")

    async def get_messages(
        self,
        session_id: str | None,
        *,
        state: dict[str, Any] | None = None,
        **kwargs: Any,
    ) -> list[Message]:
        if state is None:
            return []

        cached = state.get("messages")
        if cached:
            return list(cached)

        if not session_id:
            return []

        with get_chat_session() as db:
            rows = (
                db.query(ChatMessage)
                .filter(ChatMessage.session_id == session_id)
                .order_by(ChatMessage.id.asc())
                .all()
            )
            hydrated: list[Message] = []
            for r in rows:
                try:
                    content_obj = json.loads(r.content)
                except (ValueError, TypeError):
                    content_obj = r.content
                hydrated.append(
                    Message(
                        role=r.role,
                        contents=[content_obj] if not isinstance(content_obj, list) else content_obj,
                    )
                )

        state["messages"] = hydrated
        return list(hydrated)

    async def save_messages(
        self,
        session_id: str | None,
        messages: Sequence[Message],
        *,
        state: dict[str, Any] | None = None,
        **kwargs: Any,
    ) -> None:
        if state is None:
            return

        existing = state.get("messages", [])
        state["messages"] = [*existing, *messages]

        if not session_id:
            return

        with get_chat_session() as db:
            for m in messages:
                try:
                    content_text = json.dumps([
                        c if isinstance(c, (str, int, float, bool)) else getattr(c, "text", str(c))
                        for c in (m.contents or [])
                    ])
                except (TypeError, ValueError):
                    content_text = str(m.contents)
                db.add(ChatMessage(session_id=session_id, role=str(m.role), content=content_text))


history_provider = SqliteHistoryProvider()
print("\u2713 SqliteHistoryProvider ready")

✓ SqliteHistoryProvider ready


## 6. Middleware stack

Onion order: `InputGuardrailMiddleware` (outermost — cheapest reject) → `ExceptionHandlingAgentMiddleware` → `LoggingAgentMiddleware` (innermost — captures everything else).

In [8]:
_LOGGER_NAME = "consolidated_agent"


def _configure_logger() -> logging.Logger:
    logger = logging.getLogger(_LOGGER_NAME)
    if logger.handlers:
        return logger
    logger.setLevel(logging.INFO)
    fh = logging.FileHandler(LOG_FILE, encoding="utf-8")
    fh.setFormatter(logging.Formatter("%(asctime)s | %(levelname)s | %(message)s"))
    logger.addHandler(fh)
    logger.propagate = False
    return logger


class LoggingAgentMiddleware(AgentMiddleware):
    """Logs each agent run: inbound query, message count, duration, response snippet."""

    def __init__(self) -> None:
        self.logger = _configure_logger()

    async def process(
        self,
        context: AgentContext,
        call_next: Callable[[], Awaitable[None]],
    ) -> None:
        last = context.messages[-1] if context.messages else None
        last_text = (last.text if last and last.text else "").replace("\n", " ")[:200]
        self.logger.info(
            f"[run-start] msgs={len(context.messages or [])} last_user={last_text!r}"
        )

        start = time.perf_counter()
        try:
            await call_next()
        finally:
            duration = time.perf_counter() - start
            response_snippet = ""
            if context.result is not None:
                try:
                    response_snippet = str(getattr(context.result, "text", context.result))[:200]
                except Exception:
                    response_snippet = "<unserializable>"
            self.logger.info(
                f"[run-end] duration={duration:.3f}s response={response_snippet!r}"
            )


print("LoggingAgentMiddleware defined.")

LoggingAgentMiddleware defined.


In [9]:
class ExceptionHandlingAgentMiddleware(AgentMiddleware):
    """Catches exceptions during the run and converts them into a friendly response."""

    def __init__(self) -> None:
        self.logger = _configure_logger()

    async def process(
        self,
        context: AgentContext,
        call_next: Callable[[], Awaitable[None]],
    ) -> None:
        try:
            await call_next()
        except TimeoutError as e:
            self.logger.error(f"[exception] TimeoutError: {e}")
            context.result = AgentResponse(
                messages=[Message("assistant", [
                    "The request timed out while contacting a downstream service. "
                    "Please try again in a moment."
                ])]
            )
        except ValueError as e:
            self.logger.error(f"[exception] ValueError: {e}")
            context.result = AgentResponse(
                messages=[Message("assistant", [
                    f"I couldn't process that request because of invalid input: {e}"
                ])]
            )
        except SQLAlchemyError as e:
            self.logger.error(f"[exception] SQLAlchemyError: {e}")
            context.result = AgentResponse(
                messages=[Message("assistant", [
                    "A database error occurred while processing your request. "
                    "Our team has been notified \u2014 please try again shortly."
                ])]
            )
        except Exception as e:  # noqa: BLE001
            self.logger.exception(f"[exception] Unhandled: {e}")
            context.result = AgentResponse(
                messages=[Message("assistant", [
                    "Sorry, something went wrong while handling your request. "
                    "Please try again or contact support."
                ])]
            )


print("ExceptionHandlingAgentMiddleware defined.")

ExceptionHandlingAgentMiddleware defined.


In [10]:
class InputGuardrailMiddleware(AgentMiddleware):
    """Blocks queries containing blacklisted words (whole-word, case-insensitive)."""

    def __init__(self, blacklist_path: Path) -> None:
        self.logger = _configure_logger()
        self.words = self._load_words(blacklist_path)
        if self.words:
            pattern = r"\b(" + "|".join(re.escape(w) for w in self.words) + r")\b"
            self.regex = re.compile(pattern, re.IGNORECASE)
        else:
            self.regex = None

    @staticmethod
    def _load_words(path: Path) -> list[str]:
        if not path.exists():
            return []
        out: list[str] = []
        for line in path.read_text(encoding="utf-8").splitlines():
            stripped = line.strip()
            if not stripped or stripped.startswith("#"):
                continue
            out.append(stripped.lower())
        return out

    async def process(
        self,
        context: AgentContext,
        call_next: Callable[[], Awaitable[None]],
    ) -> None:
        if self.regex is None:
            await call_next()
            return

        last = context.messages[-1] if context.messages else None
        text = last.text if last and last.text else ""
        m = self.regex.search(text)

        if m:
            matched = m.group(1)
            session_id = getattr(context.session, "id", None) if context.session else None
            self.logger.warning(f"[guardrail] BLOCKED matched={matched!r} session={session_id}")

            try:
                with get_chat_session() as db:
                    db.add(BlockedQuery(
                        session_id=session_id,
                        query=text,
                        matched_word=matched,
                    ))
            except Exception as e:  # noqa: BLE001
                self.logger.error(f"[guardrail] failed to persist blocked query: {e}")

            refusal = (
                f"I'm sorry, I can't help with that request. Your message contained the word "
                f"'{matched}', which is on our blocked list (offensive / inappropriate / unsafe "
                "content). Please rephrase your question without that term and I'll be happy to assist."
            )
            context.result = AgentResponse(messages=[Message("assistant", [refusal])])
            raise MiddlewareTermination(result=context.result)

        await call_next()


print("InputGuardrailMiddleware defined.")

InputGuardrailMiddleware defined.


## 7. Assemble the consolidated agent

All tools (local utility, ticket management, MCP servers, FoundryAgent-as-tool), the history provider, and the middleware stack are wired into a single `FoundryChatClient.as_agent(...)` instance.

In [11]:
credential = AzureCliCredential()

client = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME"],
    credential=credential,
)

CONSOLIDATED_INSTRUCTIONS = (
    "You are a consolidated enterprise assistant for DataTech. You help with IT support tickets, "
    "travel queries, order/complaint operations, Microsoft documentation lookups, and basic "
    "utility questions. You have a strict tool-use policy.\n"
    "\n"
    "Tool-use policy (STRICT \u2014 follow without exception):\n"
    "1. IT support tickets (raise / search / resolve / close): MUST use the ticket tools "
    "(register_ticket, get_tickets_by_registered_by, get_tickets_by_resolved_by, search_tickets, "
    "resolve_ticket, close_ticket). Never fabricate ticket IDs or statuses.\n"
    "2. Travel questions (hotels, flights, packages, destinations, itineraries, tour operators, "
    "travel agencies including 'Margie\u2019s Travels'): MUST call `travel-support-agent-tool` BEFORE "
    "answering, passing the user's question verbatim as the `query` argument. Do NOT answer travel "
    "questions from your own knowledge.\n"
    "3. Microsoft / Azure / Microsoft Learn documentation questions: MUST use the Microsoft Learn "
    "MCP tool.\n"
    "4. Order or complaint operations (make order, list orders, register complaint, resolve "
    "complaint, etc.): MUST use the MAGIC v22 MCP tool.\n"
    "5. Weather questions: MUST call `get_weather`.\n"
    "6. Time questions: MUST call `get_current_time`.\n"
    "7. 'Where am I' / location questions: MUST call `get_current_location`.\n"
    "8. Only respond directly (without tools) for greetings, clarifications, or non-actionable "
    "small talk.\n"
    "\n"
    "General guidelines:\n"
    "- Address the user by name whenever it is known from the conversation.\n"
    "- After a tool call, summarize the tool's response for the user, citing IDs returned "
    "(ticket_id, order_id, complaint_id) so they can be referenced in follow-up turns.\n"
    "- Valid ticket priorities: Low, Medium, High, Critical.\n"
    "- Valid ticket statuses: Open, In Progress, On Hold, Resolved, Closed, Cancelled."
)

agent = client.as_agent(
    name="ConsolidatedAgent",
    instructions=CONSOLIDATED_INSTRUCTIONS,
    tools=[
        *ticket_tools,
        ms_learn_tool,
        magic_v22_tool,
        travel_agent_tool,
        get_weather,
        get_current_time,
        get_current_location,
    ],
    context_providers=[history_provider],
    middleware=[
        InputGuardrailMiddleware(BLACKLIST_PATH),
        ExceptionHandlingAgentMiddleware(),
        LoggingAgentMiddleware(),
    ],
)

print(f"\u2713 Agent '{agent.name}' ready with {len(ticket_tools) + 6} tools wired in.")

✓ Agent 'ConsolidatedAgent' ready with 12 tools wired in.


In [12]:
# Single shared session reused across every scenario below
session = agent.create_session()
session_id = getattr(session, "session_id", None) or getattr(session, "id", None)
print(f"Session created. id={session_id}")


async def run_streaming(query: str) -> None:
    print(f"User: {query}")
    print("Agent (streaming): ", end="", flush=True)
    async for chunk in agent.run(query, session=session, stream=True):
        if chunk.text:
            print(chunk.text, end="", flush=True)
    print("\n")


async def run_blocking(query: str) -> None:
    print(f"User: {query}")
    response = await agent.run(query, session=session)
    print(f"Agent: {response}\n")

Session created. id=1cfd3519-801d-4c1a-801d-c6e02954573c


## 8. Scenarios on the shared session

Each scenario exercises a different tool surface. Streaming and non-streaming are mixed deliberately. All scenarios share the same `session`, so context (e.g. the user's name, the ticket ID just registered, the order ID just created) flows across cells.

### Scenario A — Local utility tools (streaming)
Exercises `get_current_location`, `get_current_time`, and `get_weather` in a single turn.

In [13]:
print("=" * 60)
print("SCENARIO A \u2014 utility tools (streaming)")
print("=" * 60)

await run_streaming(
    "Hi, I'm Ramkumar. Where am I right now, what's the local time, and what's today's weather there?"
)

SCENARIO A — utility tools (streaming)
User: Hi, I'm Ramkumar. Where am I right now, what's the local time, and what's today's weather there?
Agent (streaming): Hi Ramkumar! You are currently in Bengaluru, India. 

- **Local time:** 6:20 AM on May 8, 2026.
- **Today's weather:** Cloudy with a high of 36°C. 

Let me know if there’s anything else I can assist you with!



### Scenario B — Ticket management `@tool`s (non-streaming, two turns)
Registers a High-priority VPN ticket, then lists Ramkumar's tickets to confirm the ID was persisted.

In [14]:
print("=" * 60)
print("SCENARIO B \u2014 ticket tools (non-streaming)")
print("=" * 60)

await run_blocking(
    "Please raise a High priority IT support ticket for me: my office VPN keeps dropping every "
    "10 minutes since this morning, blocking access to internal systems."
)

SCENARIO B — ticket tools (non-streaming)
User: Please raise a High priority IT support ticket for me: my office VPN keeps dropping every 10 minutes since this morning, blocking access to internal systems.
Agent: Your IT support ticket has been successfully created, Ramkumar.

- **Ticket ID:** DTKT10002
- **Description:** Office VPN keeps dropping every 10 minutes since this morning, blocking access to internal systems.
- **Priority:** High
- **Status:** Open

The IT team will address this issue shortly. Let me know if you have any updates or need further assistance!



In [15]:
await run_blocking("Show me all the tickets I have raised so far.")

User: Show me all the tickets I have raised so far.
Agent: Here is the IT support ticket you have raised so far, Ramkumar:

- **Ticket ID:** DTKT10002  
  **Description:** Office VPN keeps dropping every 10 minutes since this morning, blocking access to internal systems.  
  **Registered Date:** May 8, 2026  
  **Priority:** High  
  **Status:** Open  

Let me know if there's anything further you'd like to review or update!



### Scenario C — Public MCP server: Microsoft Learn (streaming)

In [16]:
print("=" * 60)
print("SCENARIO C \u2014 MS Learn MCP (streaming)")
print("=" * 60)

await run_streaming(
    "Use Microsoft Learn to summarize the latest guidance on configuring private endpoints "
    "for Azure OpenAI."
)

SCENARIO C — MS Learn MCP (streaming)
User: Use Microsoft Learn to summarize the latest guidance on configuring private endpoints for Azure OpenAI.
Agent (streaming): Here are the latest guidance and steps for configuring private endpoints for Azure OpenAI:

### Configuring Private Endpoints:
1. **Disable Public Network Access**: 
   - Navigate to your Azure OpenAI resource in the portal. Under the Networking tab, select "Disabled" for public access. This ensures private endpoint connections are the exclusive way to interact with the resource.

2. **Create a Private Endpoint**:
   - Visit the Private Endpoint Connections tab in the Networking section of the Azure portal.
   - Select "+ Private Endpoint" and configure the basic details such as region, subscription, and resource group.
   - Ensure the private network region and resource region align.

3. **DNS Zone Integration**: 
   - Use Azure Private DNS for network resolution.
   - Configure DNS settings with a private zone like `pri

### Scenario D — Auth-protected MCP server: MAGIC v22 (non-streaming)
Self-skips if the local `localhost:9898` server is not reachable.

In [17]:
print("=" * 60)
print("SCENARIO D \u2014 MAGIC v22 MCP (non-streaming)")
print("=" * 60)

if not MAGIC_V22_AVAILABLE:
    print("[skipped] MAGIC v22 server not reachable on localhost:9898 \u2014 start it and re-run this cell.")
else:
    await run_blocking(
        "Create a new order for customer 'Ramkumar'. Use product SKU 'LAPTOP-PRO-X1', 1 unit at "
        "$1499.99. Add a remark: 'Standard delivery'. Confirm the order_id once created, then "
        "register a HIGH priority complaint on that order on behalf of Ramkumar with the text: "
        "'The packaging arrived damaged.' Share both the order_id and complaint_id."
    )

SCENARIO D — MAGIC v22 MCP (non-streaming)
User: Create a new order for customer 'Ramkumar'. Use product SKU 'LAPTOP-PRO-X1', 1 unit at $1499.99. Add a remark: 'Standard delivery'. Confirm the order_id once created, then register a HIGH priority complaint on that order on behalf of Ramkumar with the text: 'The packaging arrived damaged.' Share both the order_id and complaint_id.
Agent: Here are the details for the actions completed:

1. **Order Created**:
   - **Order ID:** 14
   - **Order Number:** ORD10014
   - **Product SKU:** LAPTOP-PRO-X1
   - **Units:** 1
   - **Amount:** $1499.99
   - **Remarks:** Standard delivery
   - **Status:** Pending

2. **Complaint Registered**:
   - **Complaint ID:** 8
   - **Order ID:** 14
   - **Description:** The packaging arrived damaged
   - **Priority:** High
   - **Status:** Open

Let me know if there’s anything further you’d like to address regarding this order or complaint!



### Scenario E — FoundryAgent-as-a-tool: travel-support-agent (streaming)
The consolidated agent is required by its system prompt to delegate this to `travel-support-agent-tool`.

In [18]:
print("=" * 60)
print("SCENARIO E \u2014 FoundryAgent-as-tool (streaming)")
print("=" * 60)

await run_streaming("What hotels does Margie's Travels offer in Las Vegas?")

SCENARIO E — FoundryAgent-as-tool (streaming)
User: What hotels does Margie's Travels offer in Las Vegas?
Agent (streaming): Margie's Travels offers the following hotels in Las Vegas:

1. **The Volcano Hotel**: Located in the heart of The Strip, this stylish casino hotel features live entertainment and an extensive pool area.
2. **The Fountain Hotel**: A luxury accommodation option with a variety of restaurants and cocktail bars.
3. **The Canal Hotel**: An opulent Italian-themed resort offering luxurious suite accommodations.

For booking or further details, visit [Margie's Travel Website](http://www.margiestravel.com). Let me know if you need more help!



### Scenario F — Input guardrail trip (non-streaming)
We dynamically pick a real word from `db/blacklist.txt` so the demo is guaranteed to trip `InputGuardrailMiddleware`. The agent never reaches the model on this turn — the middleware short-circuits with a refusal and persists a `blocked_queries` row.

In [20]:
print("=" * 60)
print("SCENARIO F \u2014 guardrail trip (non-streaming)")
print("=" * 60)

_blacklist_words = InputGuardrailMiddleware._load_words(BLACKLIST_PATH)
if not _blacklist_words:
    print("[skipped] db/blacklist.txt is empty \u2014 add at least one word to demo the guardrail.")
else:
    trigger_word = _blacklist_words[0]
    offending_query = f"This is so frustrating, the IT admin is an {trigger_word} \u2014 please raise a ticket about it."
    await run_blocking(offending_query)

    with get_chat_session() as db:
        blocked_count = db.query(BlockedQuery).filter(BlockedQuery.session_id == session_id).count()
    print(f"blocked_queries rows for this session: {blocked_count}")

SCENARIO F — guardrail trip (non-streaming)
[skipped] db/blacklist.txt is empty — add at least one word to demo the guardrail.


### Scenario G — Exception handling (non-streaming)
Force a controlled failure inside `repository.resolve_ticket` and verify `ExceptionHandlingAgentMiddleware` returns a friendly response instead of raising.

In [21]:
print("=" * 60)
print("SCENARIO G \u2014 exception middleware (non-streaming)")
print("=" * 60)

import ticket_management.repository as _repo
_orig_resolve = _repo.resolve_ticket


def _broken_resolve(*args, **kwargs):
    raise TimeoutError("Simulated downstream timeout while resolving ticket")


_repo.resolve_ticket = _broken_resolve
try:
    await run_blocking(
        "Please resolve ticket DTKT99999 \u2014 resolver: Aarav Sharma, remarks: 'force-resolve test'."
    )
finally:
    _repo.resolve_ticket = _orig_resolve
    print("(repository.resolve_ticket restored)")

SCENARIO G — exception middleware (non-streaming)
User: Please resolve ticket DTKT99999 — resolver: Aarav Sharma, remarks: 'force-resolve test'.
Agent: I encountered an issue while attempting to resolve ticket **DTKT99999**—a timeout occurred during the process. You may retry the resolution or verify the ticket status to ensure further details. Let me know how you'd like to proceed!

(repository.resolve_ticket restored)


### Scenario H — History rehydration (non-streaming)
Without re-stating any context, ask the agent to recall the ticket ID raised in Scenario B. This proves `SqliteHistoryProvider` is replaying the session's prior turns.

In [22]:
print("=" * 60)
print("SCENARIO H \u2014 history rehydration (non-streaming)")
print("=" * 60)

await run_blocking("What was the ticket ID you registered for me earlier, and what was its priority?")

SCENARIO H — history rehydration (non-streaming)
User: What was the ticket ID you registered for me earlier, and what was its priority?
Agent: The ticket ID I registered for you earlier is **DTKT10002**, and its priority is **High**. Let me know if you need any further details or assistance!



## 9. Wrap-up — inspect the artefacts

Tail the agent log file and count rows in the chat-history tables to confirm everything was persisted.

In [23]:
print("--- Last 20 lines of agent.log ---")
if LOG_FILE.exists():
    log_lines = LOG_FILE.read_text(encoding="utf-8").splitlines()
    for line in log_lines[-20:]:
        print(line)
else:
    print("(log file not yet created)")

print("\n--- chat_history.db row counts ---")
with get_chat_session() as db:
    total_msgs = db.query(ChatMessage).count()
    session_msgs = db.query(ChatMessage).filter(ChatMessage.session_id == session_id).count()
    total_blocked = db.query(BlockedQuery).count()
    session_blocked = db.query(BlockedQuery).filter(BlockedQuery.session_id == session_id).count()

print(f"chat_messages    : total={total_msgs}, this_session={session_msgs}")
print(f"blocked_queries  : total={total_blocked}, this_session={session_blocked}")

--- Last 20 lines of agent.log ---
2026-05-08 05:55:33,684 | INFO | [run-start] msgs=1 last_user='This is so frustrating, the IT admin is an idiot — please raise a ticket about it.'
2026-05-08 05:55:37,893 | INFO | [run-end] duration=4.209s response="I'm here to assist with IT-related issues in a constructive and professional manner. If you have specific concerns or feedback about IT support, I recommend clearly outlining the issue so it can be ad"
2026-05-08 05:55:41,468 | INFO | [run-start] msgs=1 last_user="Please resolve ticket TCK-9999 with note 'force-resolve test'."
2026-05-08 05:55:48,158 | INFO | [run-end] duration=6.690s response='It seems there was an issue resolving ticket TCK-9999 due to a simulated timeout from the system. You may want to try again after some time or check the status of the ticket to ensure it is still acti'
2026-05-08 06:20:05,003 | INFO | [run-start] msgs=1 last_user="Hi, I'm Ramkumar. Where am I right now, what's the local time, and what's today's weat